In [3]:
import pandas as pd
from pathlib import Path

# Walk up from the notebook's folder until we find the repo root (the folder containing "data")
ROOT = Path.cwd()
while not (ROOT / 'data').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
print('Repo root:', ROOT)

df = pd.read_csv(ROOT / 'data' / 'processed' / 'm1_features_2021_2024.csv')
print(df.shape, df['patch_id'].nunique())   # expect (8244, 11) and 2061
print(df.isna().sum())                      # missing values per column

cols = ['NDVI', 'NDMI', 'NBR', 'valid', 'forest_frac', 'burn_frac']
print(df.groupby('year')[cols].mean().round(3))

# NDMI relative to the same-year typical patch, so the shared trend cancels
df['NDMI_rel'] = df['NDMI'] - df.groupby('year')['NDMI'].transform('median')
print(df['NDMI_rel'].describe())

# Burn flag check: how many cells are mostly burned?
one_year = df[df['year'] == 2024]
print((one_year['burn_frac'] > 0.5).sum(), 'cells more than half burned')
print((one_year['burn_frac'] == 0).sum(), 'cells with no burn')

Repo root: c:\NUST\BSCS 15D_NUST\Semester 3\AI\Project\Dryad-AI
(8244, 11) 2061
patch_id        0
row             0
col             0
year            0
NDVI           56
NDMI           56
NBR            56
NDMI_anom      56
valid           0
forest_frac     0
burn_frac       0
dtype: int64
       NDVI   NDMI    NBR  valid  forest_frac  burn_frac
year                                                    
2021  0.394 -0.095  0.055    1.0        0.713      0.546
2022  0.444 -0.044  0.148    1.0        0.713      0.546
2023  0.498  0.015  0.209    1.0        0.713      0.546
2024  0.494  0.027  0.242    1.0        0.713      0.546
count    8188.000000
mean        0.011356
std         0.107367
min        -0.253677
25%        -0.061054
50%         0.000000
75%         0.074634
max         0.390233
Name: NDMI_rel, dtype: float64
1132 cells more than half burned
777 cells with no burn


In [4]:
# 1) Are the missing cells the non-forest ones?
missing = df[df['NDVI'].isna()]
print(missing['patch_id'].nunique(), 'cells with missing indices')
print(missing['forest_frac'].describe().round(3))

# 2) Burned vs unburned: how do the indices behave across years?
df['burn_class'] = pd.cut(df['burn_frac'], bins=[-0.01, 0.05, 0.5, 1.0],
                          labels=['unburned', 'partial', 'mostly_burned'])
print(df[df['year'] == 2024]['burn_class'].value_counts())
print(df.groupby(['burn_class', 'year'], observed=True)[['NDVI', 'NDMI', 'NBR']]
        .mean().round(3))

14 cells with missing indices
count    56.0
mean      0.0
std       0.0
min       0.0
25%       0.0
50%       0.0
75%       0.0
max       0.0
Name: forest_frac, dtype: float64
burn_class
mostly_burned    1132
unburned          818
partial           111
Name: count, dtype: int64
                     NDVI   NDMI    NBR
burn_class    year                     
unburned      2021  0.488  0.007  0.236
              2022  0.503  0.009  0.238
              2023  0.549  0.059  0.286
              2024  0.531  0.046  0.280
partial       2021  0.501  0.023  0.232
              2022  0.532  0.041  0.257
              2023  0.575  0.089  0.303
              2024  0.563  0.082  0.305
mostly_burned 2021  0.317 -0.179 -0.092
              2022  0.393 -0.089  0.074
              2023  0.454 -0.024  0.146
              2024  0.460  0.008  0.208


In [5]:
import geopandas as gpd
import pandas as pd

grid = gpd.read_file(ROOT / 'data' / 'grid.geojson').to_crs(32611)
grid['cell_m2'] = grid.geometry.area
ids = gpd.read_file(ROOT / 'data' / 'raw' / 'ids_shaver_2021_2024.geojson').to_crs(32611)
ids['geometry'] = ids.geometry.make_valid()

ov = gpd.overlay(
    grid[['patch_id', 'cell_m2', 'geometry']],
    ids[['survey_year', 'dca_common_name', 'percent_mid', 'geometry']],
    how='intersection')
ov['cover'] = ov.geometry.area / ov['cell_m2']

lab = (ov.groupby(['patch_id', 'survey_year'])
         .agg(cover=('cover', 'sum'),
              max_pct=('percent_mid', 'max'),
              cause=('dca_common_name', lambda s: s.mode().iat[0]))
         .reset_index()
         .rename(columns={'survey_year': 'year'}))
lab['cover'] = lab['cover'].clip(upper=1)

df2 = df.merge(lab, on=['patch_id', 'year'], how='left')
df2['cover'] = df2['cover'].fillna(0)
df2['labelled'] = df2['cover'] >= 0.10        # polygon covers 10%+ of the cell

print('Cells with a polygon (10%+ coverage) per year:')
print(df2.groupby('year')['labelled'].sum())

# Same-year NDMI relative to the typical UNBURNED cell
med = df2[df2['burn_frac'] < 0.05].groupby('year')['NDMI'].median()
df2['NDMI_rel_unb'] = df2['NDMI'] - df2['year'].map(med)
unb = df2[(df2['burn_frac'] < 0.05) & df2['NDMI'].notna()]

print('\nUnburned cell-years:', len(unb), '| labelled:', int(unb['labelled'].sum()))
print(unb.groupby('labelled')['NDMI_rel_unb'].agg(['count', 'mean', 'median']).round(3))

print('\nLabelled cells by burn class:')
print(df2[df2['labelled']].groupby(pd.cut(df2['burn_frac'], [-0.01, 0.05, 0.5, 1.0]),
      observed=True).size())

lab.to_csv(ROOT / 'data' / 'processed' / 'm1_ids_labels_by_cell_year.csv', index=False)

Cells with a polygon (10%+ coverage) per year:
year
2021     4
2022    89
2023    49
2024     8
Name: labelled, dtype: int64

Unburned cell-years: 3220 | labelled: 98
          count   mean  median
labelled                      
False      3122  0.014  -0.005
True         98  0.148   0.154

Labelled cells by burn class:
burn_frac
(-0.01, 0.05]    98
(0.05, 0.5]      31
(0.5, 1.0]       21
dtype: int64


In [6]:
import importlib
import modules.forest_health.scoring as scoring
importlib.reload(scoring)          # picks up your edits without restarting

df = pd.read_csv(ROOT / 'data' / 'processed' / 'm1_features_2022_2026.csv')
scored, trends, bounds = scoring.score_table(df)
print('Bounds and band cutoffs:', bounds)

latest = scored[scored['year'] == scored['year'].max()]
print('\nBand counts, latest year:')
print(latest['health_band'].value_counts())

print('\nBand by burn status:')
print(pd.crosstab(latest['burn_status'], latest['health_band']))

print('\nTrend label by burn status:')
print(pd.crosstab(trends['burn_status'], trends['trend_label']))

print('\nUnburned trend slopes:')
print(trends[trends['burn_status'] == 'unburned']['trend_slope'].describe().round(2))

ModuleNotFoundError: No module named 'modules'

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.metrics import roc_auc_score

# Find the repo root (the folder that contains "data")
ROOT = Path.cwd()
while not (ROOT / 'data').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
print('Repo root:', ROOT)

sc = pd.read_csv(ROOT / 'data' / 'processed' / 'm1_health_by_patch_year.csv')
lab = pd.read_csv(ROOT / 'data' / 'processed' / 'm1_ids_labels_by_cell_year.csv')

d = sc.merge(lab, on=['patch_id', 'year'], how='left')
d['cover'] = d['cover'].fillna(0)
d['labelled'] = d['cover'] >= 0.10
d = d[d['year'].between(2022, 2024)]      # years where the polygons overlap the scored years

print('Labelled patch-years by burn status:')
print(d[d['labelled']].groupby('burn_status').size())

unb = d[(d['burn_status'] == 'unburned') & d['health_index'].notna()]
n_pos = int(unb['labelled'].sum())
print('\nUnburned scored patch-years:', len(unb), '| overlapping a USFS polygon:', n_pos)

print('\nMean / median by overlap:')
print(unb.groupby('labelled')[['health_index', 'peer_rel']].agg(['mean', 'median']).round(1))

print('\nShare in each band (rows add to 1):')
print(pd.crosstab(unb['labelled'], unb['health_band'], normalize='index').round(2))

if n_pos > 0:
    print('\nAUC (peer_rel, lower = more damaged):',
          round(roc_auc_score(unb['labelled'], -unb['peer_rel']), 2))

NameError: name 'ROOT' is not defined